# **Import des librairies et chargement des données**

In [1]:
import pandas as pd
from pathlib import Path

In [10]:
df_analyse = pd.read_csv("../data/2_interim/paquets_phrases.csv")
df_analyse.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Mais il dut prendre une chaise, la planche du ..."
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Oh! Monsieur, la religion n’a jamais fait de m..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,"De face, dans son visage séché, ses yeux garda..."
4,1893_20_Le_docteur_Pascal._clean.txt,4,s’écria la jeune fille. Mais elle était lancée...


# **Analyse préliminaire de la segmentation du corpus**

In [20]:
df_analyse.isna().sum()

nom_fichier       0
id_paquet         0
phrases_paquet    0
nb_mots_paquet    0
dtype: int64

In [24]:
df_analyse.duplicated(subset=["nom_fichier", "id_paquet"]).sum()

np.int64(0)

In [23]:
df_analyse.duplicated(subset=["phrases_paquet"]).sum()

np.int64(0)

## **affichage de la distribution du nombre de paquets par livres**

In [11]:
df_paquets = df_analyse.groupby("nom_fichier")["phrases_paquet"].count().reset_index()
df_paquets.columns = ["nom_fichier", "nb_paquets"]
df_paquets

,nom_fichier,nb_paquets
0,1865_La_confession_de_Claude._clean.txt,65
1,1866_Le_voeu_d_une_morte._clean.txt,60
2,1867_Les_mysteres_de_Marseille._clean.txt,184
3,1867_Therese_Raquin._clean.txt,80
4,1868_Madeleine_Ferat._clean.txt,124
5,1871_1_La_fortune_des_Rougon._clean.txt,154
6,1871_2_La_curee._clean.txt,123
7,1873_3_Le_ventre_de_Paris._clean.txt,136
8,1874_4_La_conquete_de_Plassans._clean.txt,165
9,1875_5_La_faute_de_l_abbe_Mouret._clean.txt,171


In [12]:
df_paquets.describe()

,nb_paquets
count,31.000000
mean,171.838710
std,52.395354
min,60.000000
25%,145.000000
50%,171.000000
75%,214.500000
max,259.000000


- Il y a un ecart type de 52. Cette écart est assez grand mais pas incohérent vu que les livres ont des tailles tres différentes. **Il y a un livre qui n'a que 60 paquets et un autre qui en a 259.**

- coefficient de variation est d’environ 30% ( écart-type / moyenne =>  52/171 ± 0.30) 

- La mediane est proche de la moyenne (171 pour la mediane et 172 pour la moyenne).

## **affichage de la distribution du nombre de mots par paquets**

In [13]:
df_phrases_mots = df_analyse["phrases_paquet"].str.split().str.len()
df_phrases_mots.describe()

count    5327.000000
mean      785.298667
std       212.129823
min        42.000000
25%       636.000000
50%       753.000000
75%       902.500000
max      2944.000000
Name: phrases_paquet, dtype: float64

- on constate qu'un paquet compte en moyenne 785 mots et la mediane compte 753 mots. Un nombre de 785 mots par paquet peut paraitre assez élévé pour du topic modeling, mais il faut garder à l'esprit que les paquets sont constitués de plusieurs phrases et que les livres de Zola sont souvent très longs. De plus, la segmentation en paquets de 5 phrases peut conduire à des paquets relativement bruités.

- l'ecart type est de 212,1. avec une dispesion **relativement faible 27%** (ecart type / moyenne => 212/785 ± 0.27).

- L’observation des quartiles indique que 25 % des paquets contiennent au maximum 636 mots, tandis que 75 % d’entre eux ne dépassent pas environ 903 mots. Malgré une certaine variabilité, la distribution reste relativement concentrée autour de la médiane, située à 753 mots. Cela est plutot un bon signe pour le topic modeling, car cela indique que la majorité des paquets ont une taille relativement homogène, ce qui peut faciliter l'identification de thèmes cohérents au sein des paquets.

- le seul point aberrant est un paquet qui contient 2944  mots, ce qui est assez élevé par rapport à la moyenne et à la médiane. Ainsi qu'un autre paquet qui ne contient seulemnt que 42 mots, ce qui est assez faible. Ce sont des valeurs extrêmes qui pourraient potentiellement influencer les résultats du topic modeling, il serait donc judicieux de les examiner de plus près pour comprendre leur nature et décider s'ils doivent être traités ou exclus de l'analyse.

## **analyse des valeurs extrêmes du corpus**

In [14]:
df_analyse["nb_mots_paquet"] = df_analyse["phrases_paquet"].str.split().str.len()
df_analyse

,nom_fichier,id_paquet,phrases_paquet,nb_mots_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...,786
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Mais il dut prendre une chaise, la planche du ...",940
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Oh! Monsieur, la religion n’a jamais fait de m...",834
3,1893_20_Le_docteur_Pascal._clean.txt,3,"De face, dans son visage séché, ses yeux garda...",760
4,1893_20_Le_docteur_Pascal._clean.txt,4,s’écria la jeune fille. Mais elle était lancée...,854
...,...,...,...,...
5322,1894_1_Lourdes._clean.txt,193,"Le diable dans cette vie si pure, dans cette â...",937
5323,1894_1_Lourdes._clean.txt,194,Des milliers de pèlerins avaient beau se rendr...,983
5324,1894_1_Lourdes._clean.txt,195,"Elle était sa maîtresse souveraine, elle le te...",814
5325,1894_1_Lourdes._clean.txt,196,Et n’aurait-il pas fallu la venue d’un nouveau...,893


In [15]:
print(f"il y'a ",(df_analyse["nb_mots_paquet"]<100).sum(), "paquets qui contiennent moins de 100 mots")

il y'a  3 paquets qui contiennent moins de 100 mots


In [16]:
print(f"il y'a ",(df_analyse["nb_mots_paquet"]>1500).sum(), "paquets qui contiennent plus de 1000 mots")

il y'a  24 paquets qui contiennent plus de 1000 mots


In [17]:
((df_analyse["nb_mots_paquet"]<100).mean())*100

np.float64(0.056316876290595085)

In [18]:
((df_analyse["nb_mots_paquet"]>1500).mean())*100

np.float64(0.4505350103247607)